<a href="https://colab.research.google.com/github/jeromelin0933/NongHui_RAG_Project/blob/feature%2Fdocx-support/%E8%BE%B2%E6%9C%83%E8%A6%8F%E7%AB%A0%E5%8A%A9%E6%89%8B_DOCX%E9%80%B2%E9%9A%8E%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q docling langchain-text-splitters langchain-community
!pip install -q sentence-transformers faiss-cpu langchain-google-genai

import antigravity
print("🚀 專案起飛：所有核心套件安裝完畢！")

In [ ]:
import os
import getpass

print("🔑 請輸入你的 Google Gemini API Key：")
os.environ["GOOGLE_API_KEY"] = getpass.getpass()

In [ ]:
import time
from docling.document_converter import DocumentConverter
from google.colab import files

# 👇 確保你已經將 DOCX 檔案上傳到 Colab 左側的檔案區
file_path = "內部規章-信用部_業務類_114.12.17.docx"

print(f"🚀 啟動 Docling，開始深度解析 {file_path} ... (請耐心等候)")
start_time = time.time()

# 1. 初始化文件轉換器並執行轉換
converter = DocumentConverter()
result = converter.convert(file_path)

# 2. 匯出為 Markdown
markdown_text = result.document.export_to_markdown()

end_time = time.time()
print(f"✅ 解析完成！總耗時: {end_time - start_time:.2f} 秒")

# 3. 備份純淨版 Markdown 供人工除錯
output_filename = "農會規章_438頁_純淨版.md"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(markdown_text)

print(f"💾 已儲存 {output_filename}，這份 Markdown 包含了完美的表格與標題結構！")

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

print("✂️ 開始進行 Markdown 結構化切片...")

# 1. 讀取剛剛 Docling 產生的 Markdown
with open("農會規章_438頁_純淨版.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

# 2. 🌟 完美對接「沈學長架構」的 Heading 階層設定
headers_to_split_on = [
    ("#", "法規標題"),  # 例如「汐止區農會信用部個人資料檔案安全維護管理辦法」
    ("##", "章"),      # 例如「第一章」
    ("###", "條")      # 例如「第一條」
]

# 3. 第一層切片：依照標題結構切分，自動保留層級為 metadata
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_text)

# 4. 第二層切片：針對過長的表格或條文，進行字數切片以符合 LLM 胃口
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
final_chunks = text_splitter.split_documents(md_header_splits)

# 5. 製作前端 UI 需要的「麵包屑 (Breadcrumb)」
for chunk in final_chunks:
    breadcrumb_parts = [chunk.metadata.get(key) for key in ["法規標題", "章", "條"] if chunk.metadata.get(key)]
    # 如果找不到任何標題，給予預設值
    chunk.metadata['breadcrumb'] = " > ".join(breadcrumb_parts) if breadcrumb_parts else "一般規章 / 無標題段落"

print(f"✅ 切片完成！從 438 頁文件中，精準切出了 {len(final_chunks)} 個知識區塊 (Chunks)。")

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS

print("🧠 正在載入 BAAI/bge-m3 地端模型...")
embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, # 使用 Colab GPU 加速
    encode_kwargs={'normalize_embeddings': True}
)

print("🗄️ 正在建立 FAISS 向量資料庫 (438頁資料量較大，約需 2~5 分鐘)...")
vectorstore = FAISS.from_documents(documents=final_chunks, embedding=embeddings)

# 儲存到本地端
vectorstore.save_local("./faiss_index")
print("✅ FAISS 資料庫建置完成！(可隨時下載 ./faiss_index 備份)")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import time

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# 📝 根據真實目錄設計的測試題
query = "關於『內部融資作業』，請問申請的額度有沒有什麼限制？或是核准的層級怎麼劃分？"

print(f"🙋 櫃員提問：{query}\n")
print("⏳ AI 思考與檢索中...\n")

# 1. 檢索
retrieved_docs = vectorstore.similarity_search(query, k=3)

# 整理上下文與麵包屑來源
formatted_texts = []
source_breadcrumbs = set()
for doc in retrieved_docs:
    formatted_texts.append(doc.page_content)
    source_breadcrumbs.add(doc.metadata.get('breadcrumb', '未知來源'))

context_text = "\n\n".join(formatted_texts)
sources_str = " / ".join(source_breadcrumbs)

# 2. Prompt 模板 (維持你原有的高標準要求)
prompt_template = ChatPromptTemplate.from_template("""
你是一位專業且嚴謹的農會信用部櫃檯助手。
請「僅根據」以下提供的【法規資料】來回答櫃員的提問。

【回答核心要求】（請嚴格遵守）：
1. 依據情境分流：針對複雜問題，請務必拆分不同情境，條列式說明。
2. 絕對不捏造：如果法規資料中沒有明確提到答案，請直接回答「根據現有內部規章，未找到相關資訊」。

【法規資料】：
{context}

【櫃員提問】：
{question}
""")

final_prompt = prompt_template.format_messages(context=context_text, question=query)

# 3. 呼叫 LLM 產生最終回答
response = llm.invoke(final_prompt)

print("✅ 成功取得 AI 回答：\n")
print(response.content)
print("\n" + "-"*40)
print(f"📑 參考法規出處：{sources_str}")